# IEEE 33-Bus Graph Transformer Dataset Pipeline (HDF5)

This notebook generates a physically valid, constraint-aware IEEE 33-bus distribution-system dataset for Graph Transformer training. It follows a constraint-aware workflow:

1. Generate bounded and structured loads.
2. Generate a radial tree-preserving switch configuration.
3. Pre-validate topology and early risk before any power-flow solve.
4. Run pandapower only on feasible candidates.
5. Stream accepted samples to HDF5.

The target labels are total losses, minimum voltage, maximum line loading, and constraint violations.

**Storage target**: `data/ieee33_dataset.h5`


In [ ]:
import json

import pandapower as pp
import pandapower.networks as pn
import networkx as nx
import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
import h5py
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

GLOBAL_SEED = 20260622
N_SAMPLES = 1000
BATCH_SIZE = 64
N_JOBS = -1
H5_PATH = "data/ieee33_dataset.h5"
COMPRESSION = "gzip"
COMPRESSION_OPTS = 4

VOLTAGE_LIMITS_PU = (0.95, 1.05)
OVERLOAD_LIMIT_PERCENT = 100.0
MAX_POWERFLOW_ATTEMPT_MULTIPLIER = 1.35

SCENARIO_POLICY = {
    "normal": 0.65,
    "mild_stress": 0.25,
    "high_stress": 0.10,
}
LOAD_DISTRIBUTION_POLICY = {
    "normal_region": {"weight": 0.80, "bounds": (0.85, 1.10)},
    "moderate_stress": {"weight": 0.15, "bounds": (1.10, 1.25)},
    "extreme_stress": {"weight": 0.05, "bounds": (1.25, 1.40)},
}
SCENARIO_LOAD_SPECS = {
    "normal": {"mean": 0.98, "std": 0.055, "clip": (0.85, 1.10), "total_bounds": (0.88, 1.08), "risk_limit": 0.88},
    "mild_stress": {"mean": 1.14, "std": 0.050, "clip": (1.05, 1.25), "total_bounds": (1.08, 1.20), "risk_limit": 0.94},
    "high_stress": {"mean": 1.28, "std": 0.055, "clip": (1.18, 1.40), "total_bounds": (1.16, 1.30), "risk_limit": 0.99},
}
MAX_CONFIG_REPEAT = 3
MIN_SWITCH_DIVERSITY = 0.30
TARGET_REJECTION_RATE = 0.20

NODE_FEATURE_NAMES = ["p_load_mw", "q_load_mvar", "feeder_depth", "distance_to_substation", "bus_type", "initial_voltage_pu"]
EDGE_FEATURE_NAMES = ["r_ohm", "x_ohm", "switch_status", "impedance_magnitude"]
GLOBAL_FEATURE_NAMES = ["total_load_mw", "num_buses", "num_lines", "active_lines"]
LABEL_NAMES = ["total_loss_kw", "min_voltage_pu", "max_line_loading_percent", "voltage_violation_count", "overload_count", "constraint_violation_count"]

np.random.seed(GLOBAL_SEED)
plt.rcParams["figure.figsize"] = (9, 5)
print(f"Configuration ready | target samples={N_SAMPLES:,} | batch_size={BATCH_SIZE} | seed={GLOBAL_SEED}")


## STEP 1 — Load IEEE 33 Bus System

`load_ieee33()` uses `pandapower.networks.case33bw()` and prepares the standard normally-open tie branches when needed. The base network is solved once and serialized to JSON so worker samples can clone a warm-started network without extra libraries.


In [ ]:
STANDARD_TIE_BRANCH_DATA = [
    {"from_bus": 20, "to_bus": 7, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 8, "to_bus": 14, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 11, "to_bus": 21, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 17, "to_bus": 32, "r_ohm": 0.5, "x_ohm": 0.5},
    {"from_bus": 24, "to_bus": 28, "r_ohm": 0.5, "x_ohm": 0.5},
]

BASE_NET_JSON = None
BASE_GRAPH = None
BASE_POWERFLOW_RESULT = None
BASE_TOTAL_LOAD_MW = None


def _ensure_line_columns(net):
    if "in_service" not in net.line.columns:
        net.line["in_service"] = True
    net.line["in_service"] = net.line["in_service"].fillna(True).astype(bool)
    if "is_tie" not in net.line.columns:
        net.line["is_tie"] = False
    net.line["is_tie"] = net.line["is_tie"].fillna(False).astype(bool)


def _line_mask_for_endpoints(net, from_bus, to_bus):
    from_values = net.line["from_bus"].astype(int).to_numpy()
    to_values = net.line["to_bus"].astype(int).to_numpy()
    return ((from_values == int(from_bus)) & (to_values == int(to_bus))) | ((from_values == int(to_bus)) & (to_values == int(from_bus)))


def _median_or_default(series, default_value):
    if series is None or len(series) == 0:
        return float(default_value)
    value = float(pd.Series(series).dropna().median())
    return value if np.isfinite(value) else float(default_value)


def _normalize_line_current_limits(net, nominal_max_i_ka=0.40):
    if "max_i_ka" not in net.line.columns:
        net.line["max_i_ka"] = nominal_max_i_ka
    max_i = net.line["max_i_ka"].astype(float)
    invalid = (~np.isfinite(max_i)) | (max_i <= 0.0) | (max_i > 5.0)
    net.line.loc[invalid, "max_i_ka"] = float(nominal_max_i_ka)
    return net


def _ensure_ieee33_tie_lines(net):
    _ensure_line_columns(net)
    max_i_ka = _median_or_default(net.line["max_i_ka"] if "max_i_ka" in net.line.columns else None, 0.40)
    c_nf_per_km = _median_or_default(net.line["c_nf_per_km"] if "c_nf_per_km" in net.line.columns else None, 0.0)
    for tie in STANDARD_TIE_BRANCH_DATA:
        mask = _line_mask_for_endpoints(net, tie["from_bus"], tie["to_bus"])
        matching_indices = list(net.line.index[mask])
        if matching_indices:
            net.line.loc[matching_indices, "is_tie"] = True
            net.line.loc[matching_indices, "in_service"] = False
            continue
        line_idx = pp.create_line_from_parameters(
            net,
            from_bus=tie["from_bus"],
            to_bus=tie["to_bus"],
            length_km=1.0,
            r_ohm_per_km=tie["r_ohm"],
            x_ohm_per_km=tie["x_ohm"],
            c_nf_per_km=c_nf_per_km,
            max_i_ka=max_i_ka,
            name=f"tie_{tie['from_bus'] + 1}_{tie['to_bus'] + 1}",
            in_service=False,
        )
        net.line.at[line_idx, "is_tie"] = True
    net.line["in_service"] = net.line["in_service"].astype(bool)
    net.line["is_tie"] = net.line["is_tie"].astype(bool)
    return net


def get_root_bus(net):
    if len(net.ext_grid) > 0 and "bus" in net.ext_grid.columns:
        return int(net.ext_grid.iloc[0]["bus"])
    return int(net.bus.index.min())


def load_ieee33():
    net = pn.case33bw()
    _ensure_ieee33_tie_lines(net)
    _normalize_line_current_limits(net, nominal_max_i_ka=0.40)
    base_graph = build_graph(net, active_only=True)
    if not validate_configuration(net):
        raise ValueError("Prepared IEEE-33 base configuration is not radial and connected.")
    return net, base_graph


## STEP 2 — Graph Builder and STEP 3 — Topology Metrics

The graph builder creates node and edge attributes from the pandapower network. Topology metrics use BFS and impedance-weighted shortest paths from the substation/root bus.


In [ ]:
def _bus_load_table(net):
    loads = pd.DataFrame(index=net.bus.index, data={"p_load": 0.0, "q_load": 0.0})
    if len(net.load) == 0:
        return loads
    load_df = net.load.copy()
    if "in_service" in load_df.columns:
        load_df = load_df[load_df["in_service"].fillna(True).astype(bool)]
    if len(load_df) == 0:
        return loads
    grouped = load_df.groupby("bus")[["p_mw", "q_mvar"]].sum()
    loads.loc[grouped.index, "p_load"] = grouped["p_mw"].astype(float)
    loads.loc[grouped.index, "q_load"] = grouped["q_mvar"].astype(float)
    return loads


def _initial_voltage_pu_table(net):
    voltage = pd.Series(1.0, index=net.bus.index, dtype=float)
    if len(net.ext_grid) > 0:
        for _, row in net.ext_grid.iterrows():
            vm_pu = float(row["vm_pu"]) if "vm_pu" in net.ext_grid.columns else 1.0
            voltage.loc[int(row["bus"])] = vm_pu
    return voltage


def line_is_closed(net, line_idx):
    closed = bool(net.line.at[line_idx, "in_service"])
    if "switch" in net and len(net.switch) > 0:
        switches = net.switch[(net.switch["et"] == "l") & (net.switch["element"].astype(int) == int(line_idx))]
        if len(switches) > 0:
            closed = closed and bool(switches["closed"].fillna(True).astype(bool).all())
    return bool(closed)


def _line_impedance(net, line_idx):
    row = net.line.loc[line_idx]
    length = float(row["length_km"]) if "length_km" in net.line.columns else 1.0
    r = float(row["r_ohm_per_km"]) * length if "r_ohm_per_km" in net.line.columns else 0.0
    x = float(row["x_ohm_per_km"]) * length if "x_ohm_per_km" in net.line.columns else 0.0
    return r, x


def build_graph(net, active_only=False):
    _ensure_line_columns(net)
    G = nx.Graph()
    load_table = _bus_load_table(net)
    voltage_table = _initial_voltage_pu_table(net)
    root_bus = get_root_bus(net)
    for bus_idx in net.bus.index:
        bus_idx_int = int(bus_idx)
        G.add_node(
            bus_idx_int,
            p_load=float(load_table.loc[bus_idx, "p_load"]),
            q_load=float(load_table.loc[bus_idx, "q_load"]),
            bus_type=1.0 if bus_idx_int == root_bus else 0.0,
            voltage=float(voltage_table.loc[bus_idx]),
        )
    for line_idx, row in net.line.iterrows():
        switch_status = 1.0 if line_is_closed(net, line_idx) else 0.0
        if active_only and switch_status < 0.5:
            continue
        r, x = _line_impedance(net, line_idx)
        G.add_edge(
            int(row["from_bus"]),
            int(row["to_bus"]),
            line_idx=int(line_idx),
            r=float(r),
            x=float(x),
            impedance=float(np.sqrt(r * r + x * x)),
            switch_status=float(switch_status),
            is_tie=bool(row["is_tie"]),
        )
    return G


def compute_feeder_depth(G, root_bus):
    depth = {int(node): -1 for node in G.nodes}
    if root_bus not in G.nodes:
        return depth
    for node, value in nx.single_source_shortest_path_length(G, int(root_bus)).items():
        depth[int(node)] = int(value)
    return depth


def compute_distance_to_substation(G, root_bus):
    distance = {int(node): np.inf for node in G.nodes}
    if root_bus not in G.nodes:
        return distance
    try:
        lengths = nx.single_source_dijkstra_path_length(G, int(root_bus), weight="impedance")
    except Exception:
        lengths = nx.single_source_shortest_path_length(G, int(root_bus))
    for node, value in lengths.items():
        distance[int(node)] = float(value)
    return distance


## STEP 4 — Load Scenario Generator, STEP 5 — Switch Configuration Generator, and STEP 6 — Validation Layer

Loads are generated from bounded clipped normal distributions rather than naive uniform sampling. Switches use one radial branch exchange: close one normally-open tie, identify the induced cycle, and open one active branch from that cycle. Invalid topologies are rejected immediately before power flow.


In [ ]:
def check_connectivity(G):
    return len(G.nodes) > 0 and nx.is_connected(G)


def check_radiality(G):
    return len(G.nodes) > 0 and nx.is_tree(G)


def validate_configuration(net):
    active_graph = build_graph(net, active_only=True)
    return check_connectivity(active_graph) and check_radiality(active_graph)


def prevalidate_before_powerflow(net):
    active_graph = build_graph(net, active_only=True)
    connected = check_connectivity(active_graph)
    radial = check_radiality(active_graph)
    return {
        "valid": bool(connected and radial),
        "connected": bool(connected),
        "radial": bool(radial),
        "nodes": int(active_graph.number_of_nodes()),
        "active_edges": int(active_graph.number_of_edges()),
    }


def _choose_distribution_region(rng):
    names = list(LOAD_DISTRIBUTION_POLICY.keys())
    weights = np.array([LOAD_DISTRIBUTION_POLICY[name]["weight"] for name in names], dtype=float)
    weights = weights / weights.sum()
    return str(rng.choice(names, p=weights))


def _bounded_normal(rng, mean, std, low, high, size):
    values = rng.normal(loc=mean, scale=std, size=size)
    return np.clip(values, low, high)


def _enforce_total_load_bounds(scales, load_buses, base_bus_load, total_bounds, clip_bounds):
    low_total, high_total = total_bounds
    low_clip, high_clip = clip_bounds
    weights = np.array([base_bus_load.get(int(bus), 0.0) for bus in load_buses], dtype=float)
    if weights.sum() <= 0:
        return np.clip(scales, low_clip, high_clip)
    weighted_average = float(np.sum(scales * weights) / np.sum(weights))
    adjusted = scales.copy()
    if weighted_average > high_total:
        adjusted *= high_total / weighted_average
    elif weighted_average < low_total:
        adjusted *= low_total / max(weighted_average, 1e-9)
    return np.clip(adjusted, low_clip, high_clip)


def generate_feasible_loads(net, seed, scenario_class=None):
    rng = np.random.default_rng(int(seed))
    if scenario_class is None:
        region = _choose_distribution_region(rng)
        if region == "normal_region":
            scenario_class = "normal"
        elif region == "moderate_stress":
            scenario_class = "mild_stress"
        else:
            scenario_class = "high_stress"
    if scenario_class not in SCENARIO_LOAD_SPECS:
        raise ValueError(f"Unknown scenario_class={scenario_class}")

    spec = SCENARIO_LOAD_SPECS[scenario_class]
    scales = pd.Series(1.0, index=net.bus.index, dtype=float)
    load_table_before = _bus_load_table(net)
    base_bus_load = {int(idx): float(load_table_before.loc[idx, "p_load"]) for idx in load_table_before.index}
    if len(net.load) == 0:
        return scales.to_numpy(dtype=np.float32), {"scenario_class": scenario_class, "total_load_ratio": 1.0}

    load_buses = sorted(pd.unique(net.load["bus"].astype(int)))
    raw_scales = _bounded_normal(
        rng,
        mean=spec["mean"],
        std=spec["std"],
        low=spec["clip"][0],
        high=spec["clip"][1],
        size=len(load_buses),
    )
    bus_scales = _enforce_total_load_bounds(raw_scales, load_buses, base_bus_load, spec["total_bounds"], spec["clip"])
    for bus_idx, scale in zip(load_buses, bus_scales):
        scales.loc[int(bus_idx)] = float(scale)
    for load_idx, row in net.load.iterrows():
        scale = float(scales.loc[int(row["bus"])])
        net.load.at[load_idx, "p_mw"] = float(row["p_mw"]) * scale
        net.load.at[load_idx, "q_mvar"] = float(row["q_mvar"]) * scale

    load_after = _bus_load_table(net)
    base_total = max(float(load_table_before["p_load"].sum()), 1e-9)
    total_ratio = float(load_after["p_load"].sum() / base_total)
    return scales.to_numpy(dtype=np.float32), {
        "scenario_class": scenario_class,
        "distribution": "bounded_clipped_normal",
        "total_load_ratio": total_ratio,
        "total_bounds": list(spec["total_bounds"]),
        "clip_bounds": list(spec["clip"]),
    }


def graph_signature(net):
    edge_list = []
    for line_idx, row in net.line.iterrows():
        if line_is_closed(net, line_idx):
            edge_list.append(tuple(sorted((int(row["from_bus"]), int(row["to_bus"])))) )
    return str(abs(hash(tuple(sorted(edge_list)))))


def switch_config_key(net):
    return ",".join(str(int(idx)) for idx in net.line.index if line_is_closed(net, idx))


def _cycle_edges_from_nodes(net, cycle_nodes):
    cycle_set = set(int(node) for node in cycle_nodes)
    cycle_edges = []
    for line_idx, row in net.line.iterrows():
        if not line_is_closed(net, line_idx):
            continue
        if int(row["from_bus"]) in cycle_set and int(row["to_bus"]) in cycle_set:
            cycle_edges.append(int(line_idx))
    return cycle_edges


def generate_radial_switch_configuration(net, seed=None, max_attempts=80):
    rng = np.random.default_rng(GLOBAL_SEED if seed is None else int(seed))
    original_status = net.line["in_service"].astype(bool).copy()
    open_ties = [int(idx) for idx in net.line.index if bool(net.line.at[idx, "is_tie"]) and not line_is_closed(net, idx)]
    if len(open_ties) == 0:
        raise ValueError("No normally-open tie branches are available.")

    base_graph = build_graph(net, active_only=True)
    base_depth = compute_feeder_depth(base_graph, get_root_bus(net))
    for attempt in range(max_attempts):
        net.line["in_service"] = original_status.copy()
        close_line = int(rng.choice(open_ties))
        net.line.at[close_line, "in_service"] = True
        trial_graph = build_graph(net, active_only=True)
        cycles = nx.cycle_basis(trial_graph)
        if len(cycles) == 0:
            continue
        close_row = net.line.loc[close_line]
        close_endpoints = {int(close_row["from_bus"]), int(close_row["to_bus"])}
        selected_cycle = cycles[0]
        for cycle in cycles:
            if close_endpoints.issubset(set(int(node) for node in cycle)):
                selected_cycle = cycle
                break
        cycle_edges = _cycle_edges_from_nodes(net, selected_cycle)
        candidates = [idx for idx in cycle_edges if idx != close_line and not bool(net.line.at[idx, "is_tie"])]
        if len(candidates) == 0:
            continue
        candidate_scores = []
        for line_idx in candidates:
            row = net.line.loc[line_idx]
            endpoint_depths = [base_depth.get(int(row["from_bus"]), 0), base_depth.get(int(row["to_bus"]), 0)]
            candidate_scores.append(max(endpoint_depths) + 1.0)
        inverse_scores = 1.0 / np.asarray(candidate_scores, dtype=float)
        probabilities = inverse_scores / inverse_scores.sum()
        open_line = int(rng.choice(candidates, p=probabilities))
        net.line.at[open_line, "in_service"] = False
        check = prevalidate_before_powerflow(net)
        if check["valid"]:
            open_row = net.line.loc[open_line]
            cycle_length = int(len(selected_cycle))
            disruption = float(cycle_length / max(len(net.bus), 1))
            return {
                "closed_tie_line": int(close_line),
                "closed_tie_edge": [int(close_row["from_bus"]), int(close_row["to_bus"])],
                "opened_branch_line": int(open_line),
                "opened_branch_edge": [int(open_row["from_bus"]), int(open_row["to_bus"])],
                "cycle_length": cycle_length,
                "switch_disruption_magnitude": disruption,
                "attempts": int(attempt + 1),
                "topology_id": graph_signature(net),
                "switch_config_key": switch_config_key(net),
            }
    net.line["in_service"] = original_status.copy()
    raise ValueError(f"Failed to generate a radial branch exchange after {max_attempts} attempts.")


def estimate_risk_score(net, load_metadata, switch_metadata):
    active_graph = build_graph(net, active_only=True)
    root_bus = get_root_bus(net)
    depth = compute_feeder_depth(active_graph, root_bus)
    load_table = _bus_load_table(net)
    depths = np.array([max(depth.get(int(bus), 0), 0) for bus in load_table.index], dtype=float)
    loads = load_table["p_load"].to_numpy(dtype=float)
    if loads.sum() > 0 and depths.max() > 0:
        depth_imbalance = float(np.sum(loads * depths) / (loads.sum() * depths.max()))
    else:
        depth_imbalance = 0.0
    total_load_ratio = float(load_metadata.get("total_load_ratio", 1.0))
    load_risk = float(np.clip((total_load_ratio - 0.90) / 0.45, 0.0, 1.0))
    switch_risk = float(np.clip(switch_metadata.get("switch_disruption_magnitude", 0.0), 0.0, 1.0))
    risk_score = 0.55 * load_risk + 0.25 * depth_imbalance + 0.20 * switch_risk
    return {
        "risk_score": float(risk_score),
        "load_risk": load_risk,
        "depth_imbalance": depth_imbalance,
        "switch_risk": switch_risk,
    }


## STEP 7 — Power Flow Engine and STEP 8 — Feature Extraction

Power flow is called only after topology and early-risk checks. The base solved network is cached and cloned through pandapower JSON serialization, enabling warm starts without adding non-approved libraries.


In [ ]:
def _has_powerflow_initialization(net):
    return (
        hasattr(net, "res_bus")
        and len(net.res_bus) == len(net.bus)
        and "vm_pu" in net.res_bus.columns
        and np.isfinite(net.res_bus["vm_pu"].to_numpy(dtype=float)).all()
    )


def run_powerflow(net):
    topology_check = prevalidate_before_powerflow(net)
    if not topology_check["valid"]:
        return {"success": False, "reason": "pre_powerflow_invalid_topology", "topology_check": topology_check}
    init_mode = "results" if _has_powerflow_initialization(net) else "flat"
    try:
        pp.runpp(
            net,
            algorithm="nr",
            init=init_mode,
            calculate_voltage_angles=False,
            enforce_q_lims=False,
            max_iteration=50,
            tolerance_mva=1e-8,
            numba=False,
        )
    except Exception as first_exc:
        if init_mode == "results":
            try:
                pp.runpp(
                    net,
                    algorithm="nr",
                    init="flat",
                    calculate_voltage_angles=False,
                    enforce_q_lims=False,
                    max_iteration=50,
                    tolerance_mva=1e-8,
                    numba=False,
                )
            except Exception as second_exc:
                return {"success": False, "reason": f"powerflow_failed: {type(second_exc).__name__}: {second_exc}"}
        else:
            return {"success": False, "reason": f"powerflow_failed: {type(first_exc).__name__}: {first_exc}"}
    if not bool(getattr(net, "converged", False)) and init_mode == "results":
        try:
            pp.runpp(
                net,
                algorithm="nr",
                init="flat",
                calculate_voltage_angles=False,
                enforce_q_lims=False,
                max_iteration=50,
                tolerance_mva=1e-8,
                numba=False,
            )
        except Exception as exc:
            return {"success": False, "reason": f"powerflow_failed: {type(exc).__name__}: {exc}"}

    if not bool(getattr(net, "converged", False)):
        return {"success": False, "reason": "powerflow_not_converged"}

    total_loss_kw = 0.0
    if "res_line" in net and len(net.res_line) > 0 and "pl_mw" in net.res_line.columns:
        total_loss_kw += float(np.nan_to_num(net.res_line["pl_mw"].to_numpy(dtype=float), nan=0.0).sum() * 1000.0)
    vm_pu = np.nan_to_num(net.res_bus["vm_pu"].to_numpy(dtype=float), nan=np.nan)
    loading = np.array([0.0], dtype=float)
    if "res_line" in net and len(net.res_line) > 0 and "loading_percent" in net.res_line.columns:
        loading = np.nan_to_num(net.res_line["loading_percent"].to_numpy(dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    v_min, v_max = VOLTAGE_LIMITS_PU
    voltage_violation_count = int(((vm_pu < v_min) | (vm_pu > v_max)).sum())
    overload_count = int((loading > OVERLOAD_LIMIT_PERCENT).sum())
    return {
        "success": True,
        "total_loss_kw": float(total_loss_kw),
        "min_voltage_pu": float(np.nanmin(vm_pu)),
        "max_line_loading_percent": float(np.nanmax(loading)),
        "voltage_violation_count": voltage_violation_count,
        "overload_count": overload_count,
        "constraint_violation_count": int(voltage_violation_count + overload_count),
    }


def initialize_base_network():
    global BASE_NET_JSON, BASE_GRAPH, BASE_POWERFLOW_RESULT, BASE_TOTAL_LOAD_MW
    base_net, BASE_GRAPH = load_ieee33()
    BASE_TOTAL_LOAD_MW = float(_bus_load_table(base_net)["p_load"].sum())
    BASE_POWERFLOW_RESULT = run_powerflow(base_net)
    if not BASE_POWERFLOW_RESULT.get("success", False):
        raise RuntimeError(f"Base IEEE-33 power flow failed: {BASE_POWERFLOW_RESULT}")
    BASE_NET_JSON = pp.to_json(base_net)
    print(
        "Base IEEE-33 cached | "
        f"buses={len(base_net.bus)} | lines={len(base_net.line)} | "
        f"base_load={BASE_TOTAL_LOAD_MW:.3f} MW | "
        f"base_min_v={BASE_POWERFLOW_RESULT['min_voltage_pu']:.4f} | "
        f"base_max_loading={BASE_POWERFLOW_RESULT['max_line_loading_percent']:.2f}%"
    )
    return base_net, BASE_GRAPH


def clone_base_network():
    if BASE_NET_JSON is None:
        initialize_base_network()
    return pp.from_json_string(BASE_NET_JSON)


def extract_graph_features(net, G=None):
    if G is None:
        G = build_graph(net, active_only=False)
    active_graph = build_graph(net, active_only=True)
    root_bus = get_root_bus(net)
    depth = compute_feeder_depth(active_graph, root_bus)
    distance = compute_distance_to_substation(active_graph, root_bus)
    load_table = _bus_load_table(net)
    voltage_table = _initial_voltage_pu_table(net)

    node_features = []
    for bus_idx in net.bus.index:
        bus_idx_int = int(bus_idx)
        node_features.append([
            float(load_table.loc[bus_idx, "p_load"]),
            float(load_table.loc[bus_idx, "q_load"]),
            float(depth.get(bus_idx_int, -1)),
            float(distance.get(bus_idx_int, np.inf)),
            1.0 if bus_idx_int == root_bus else 0.0,
            float(voltage_table.loc[bus_idx]),
        ])

    edge_features = []
    for line_idx, _ in net.line.iterrows():
        r, x = _line_impedance(net, line_idx)
        edge_features.append([
            float(r),
            float(x),
            1.0 if line_is_closed(net, line_idx) else 0.0,
            float(np.sqrt(r * r + x * x)),
        ])

    global_features = np.array([
        float(load_table["p_load"].sum()),
        float(len(net.bus)),
        float(len(net.line)),
        float(sum(1 for idx in net.line.index if line_is_closed(net, idx))),
    ], dtype=np.float32)
    return {
        "node_features": np.asarray(node_features, dtype=np.float32),
        "edge_features": np.asarray(edge_features, dtype=np.float32),
        "global_features": global_features,
    }


## STEP 9 — HDF5 Streaming Storage, STEP 10 — Sample Generation Pipeline, and STEP 11 — Parallel Generation

Workers generate candidates, but HDF5 writes are performed only by the notebook process. This avoids concurrent HDF5 corruption and scales to large datasets because graph tensors are streamed batch-by-batch.


In [ ]:
def initialize_hdf5(path, overwrite=True):
    mode = "w" if overwrite else "a"
    with h5py.File(path, mode, track_order=True) as h5:
        h5.attrs["schema_version"] = "constraint_aware_flat_v1"
        h5.attrs["dataset_name"] = "ieee33_graph_transformer_constraint_aware_dataset"
        h5.attrs["global_seed"] = int(GLOBAL_SEED)
        h5.attrs["node_feature_names"] = json.dumps(NODE_FEATURE_NAMES)
        h5.attrs["edge_feature_names"] = json.dumps(EDGE_FEATURE_NAMES)
        h5.attrs["global_feature_names"] = json.dumps(GLOBAL_FEATURE_NAMES)
        h5.attrs["label_names"] = json.dumps(LABEL_NAMES)
        h5.attrs["scenario_policy"] = json.dumps(SCENARIO_POLICY)
        h5.attrs["load_distribution_policy"] = json.dumps(LOAD_DISTRIBUTION_POLICY)
        h5.attrs["total_attempted"] = 0
        h5.attrs["valid_samples"] = 0
        h5.attrs["rejected_samples"] = 0
        h5.attrs["powerflow_calls"] = 0
        h5.attrs["rejection_reasons"] = json.dumps({})
    return path


def _write_array(group, name, value):
    group.create_dataset(
        name,
        data=np.asarray(value),
        compression=COMPRESSION,
        compression_opts=COMPRESSION_OPTS,
        shuffle=True,
    )


def write_records_to_hdf5(path, records, start_sample_id):
    string_dtype = h5py.string_dtype(encoding="utf-8")
    next_id = int(start_sample_id)
    with h5py.File(path, "a", track_order=True) as h5:
        for record in records:
            group = h5.create_group(f"sample_{next_id:06d}")
            _write_array(group, "node_features", record["node_features"])
            _write_array(group, "edge_features", record["edge_features"])
            _write_array(group, "global_features", record["global_features"])
            _write_array(group, "labels", record["labels"])
            group.create_dataset("metadata", data=json.dumps(record["metadata"]), dtype=string_dtype)
            group.attrs["seed"] = int(record["seed"])
            group.attrs["scenario_class"] = record["metadata"]["scenario_class"]
            group.attrs["switch_config_key"] = record["metadata"]["switch_config_key"]
            group.attrs["topology_id"] = record["metadata"]["topology_id"]
            group.attrs["risk_score"] = float(record["metadata"]["risk"]["risk_score"])
            group.attrs["total_load_mw"] = float(record["global_features"][0])
            group.attrs["min_voltage_pu"] = float(record["labels"][1])
            group.attrs["max_line_loading_percent"] = float(record["labels"][2])
            next_id += 1
    return next_id


def update_hdf5_report(path, state):
    with h5py.File(path, "a") as h5:
        h5.attrs["total_attempted"] = int(state["total_attempted"])
        h5.attrs["valid_samples"] = int(state["valid_samples"])
        h5.attrs["rejected_samples"] = int(state["rejected_samples"])
        h5.attrs["powerflow_calls"] = int(state["powerflow_calls"])
        h5.attrs["rejection_reasons"] = json.dumps(state["rejection_reasons"])
        h5.attrs["switch_diversity_score"] = float(len(state["config_counts"]) / max(state["valid_samples"], 1))
        h5.attrs["base_powerflow_cached"] = bool(BASE_POWERFLOW_RESULT is not None and BASE_POWERFLOW_RESULT.get("success", False))


def _post_powerflow_filter(pf_result, scenario_class):
    if not pf_result.get("success", False):
        return False, pf_result.get("reason", "powerflow_failed")
    min_voltage = float(pf_result["min_voltage_pu"])
    max_loading = float(pf_result["max_line_loading_percent"])
    if min_voltage < 0.68:
        return False, "voltage_collapse_filter"
    if max_loading > 250.0:
        return False, "extreme_overload_filter"
    return True, "accepted"


def _build_success_record(seed, net, load_scaling, load_metadata, switch_metadata, topology_check, risk, pf_result):
    features = extract_graph_features(net, build_graph(net, active_only=False))
    labels = np.array([
        pf_result["total_loss_kw"],
        pf_result["min_voltage_pu"],
        pf_result["max_line_loading_percent"],
        pf_result["voltage_violation_count"],
        pf_result["overload_count"],
        pf_result["constraint_violation_count"],
    ], dtype=np.float32)
    metadata = {
        "seed": int(seed),
        "scenario_class": load_metadata["scenario_class"],
        "load_scaling_vector": [float(value) for value in load_scaling],
        "load_metadata": load_metadata,
        "switch_operations": switch_metadata,
        "switch_config_key": switch_metadata["switch_config_key"],
        "topology_id": switch_metadata["topology_id"],
        "risk": risk,
        "topology_prevalidation": topology_check,
    }
    return {
        "success": True,
        "seed": int(seed),
        "node_features": features["node_features"],
        "edge_features": features["edge_features"],
        "global_features": features["global_features"],
        "labels": labels,
        "metadata": metadata,
        "powerflow_calls": 1,
    }


def generate_sample(seed, scenario_class=None, history_snapshot=None, internal_attempts=6):
    history_snapshot = history_snapshot or {"config_counts": {}}
    last_reason = "uninitialized"
    powerflow_calls = 0
    for attempt in range(int(internal_attempts)):
        attempt_seed = int(seed) + attempt * 1009
        try:
            net = clone_base_network()
            load_scaling, load_metadata = generate_feasible_loads(net, seed=attempt_seed + 101, scenario_class=scenario_class)
            scenario_class = load_metadata["scenario_class"]
            switch_metadata = generate_radial_switch_configuration(net, seed=attempt_seed + 202)
            topology_check = prevalidate_before_powerflow(net)
            if not topology_check["valid"]:
                last_reason = "pre_powerflow_invalid_topology"
                continue

            risk = estimate_risk_score(net, load_metadata, switch_metadata)
            risk_limit = float(SCENARIO_LOAD_SPECS[scenario_class]["risk_limit"])
            if risk["risk_score"] > risk_limit:
                last_reason = "early_risk_filter"
                continue

            config_key = switch_metadata["switch_config_key"]
            if int(history_snapshot.get("config_counts", {}).get(config_key, 0)) >= MAX_CONFIG_REPEAT:
                last_reason = "switch_config_repeat_filter"
                continue

            pf_result = run_powerflow(net)
            powerflow_calls += 1
            post_ok, post_reason = _post_powerflow_filter(pf_result, scenario_class)
            if not post_ok:
                last_reason = post_reason
                continue
            record = _build_success_record(seed, net, load_scaling, load_metadata, switch_metadata, topology_check, risk, pf_result)
            record["powerflow_calls"] = powerflow_calls
            record["internal_attempts"] = int(attempt + 1)
            return record
        except Exception as exc:
            last_reason = f"sample_failed: {type(exc).__name__}: {exc}"
    return {"success": False, "seed": int(seed), "reason": last_reason, "powerflow_calls": powerflow_calls}


def _scenario_deficits(state, target_samples):
    target_counts = {name: int(round(weight * target_samples)) for name, weight in SCENARIO_POLICY.items()}
    delta = target_samples - sum(target_counts.values())
    target_counts["normal"] += delta
    accepted_counts = state["scenario_counts"]
    return {name: target_counts[name] - accepted_counts.get(name, 0) for name in target_counts}


def _choose_scenario_class(state, target_samples, rng):
    deficits = _scenario_deficits(state, target_samples)
    positive = {name: max(count, 0) for name, count in deficits.items()}
    if sum(positive.values()) <= 0:
        names = list(SCENARIO_POLICY.keys())
        weights = np.array([SCENARIO_POLICY[name] for name in names], dtype=float)
    else:
        names = list(positive.keys())
        weights = np.array([positive[name] for name in names], dtype=float)
    weights = weights / weights.sum()
    return str(rng.choice(names, p=weights))


def generate_dataset(n_samples=N_SAMPLES, h5_path=H5_PATH, batch_size=BATCH_SIZE, n_jobs=N_JOBS, overwrite=True):
    initialize_base_network()
    h5_path = initialize_hdf5(h5_path, overwrite=overwrite)
    rng = np.random.default_rng(GLOBAL_SEED)
    state = {
        "total_attempted": 0,
        "valid_samples": 0,
        "rejected_samples": 0,
        "powerflow_calls": 0,
        "rejection_reasons": {},
        "config_counts": {},
        "scenario_counts": {name: 0 for name in SCENARIO_POLICY},
    }
    next_seed = int(GLOBAL_SEED)
    next_sample_id = 1
    max_attempts = int(np.ceil(n_samples * MAX_POWERFLOW_ATTEMPT_MULTIPLIER))
    progress = tqdm(total=n_samples, desc="Accepted IEEE-33 samples", unit="sample")

    while state["valid_samples"] < n_samples and state["total_attempted"] < max_attempts:
        remaining = n_samples - state["valid_samples"]
        candidate_count = min(batch_size, max(remaining * 2, 1))
        history_snapshot = {"config_counts": dict(state["config_counts"])}
        plans = []
        for _ in range(candidate_count):
            scenario_class = _choose_scenario_class(state, n_samples, rng)
            plans.append((next_seed, scenario_class, history_snapshot))
            next_seed += 1
        results = Parallel(n_jobs=n_jobs, backend="loky")(
            delayed(generate_sample)(seed, scenario_class, snapshot)
            for seed, scenario_class, snapshot in plans
        )
        accepted_records = []
        for result in results:
            if state["valid_samples"] >= n_samples:
                break
            state["total_attempted"] += 1
            state["powerflow_calls"] += int(result.get("powerflow_calls", 0))
            if result.get("success", False):
                config_key = result["metadata"]["switch_config_key"]
                if int(state["config_counts"].get(config_key, 0)) >= MAX_CONFIG_REPEAT:
                    state["rejected_samples"] += 1
                    reason = "switch_config_repeat_filter"
                    state["rejection_reasons"][reason] = int(state["rejection_reasons"].get(reason, 0)) + 1
                    continue
                state["config_counts"][config_key] = int(state["config_counts"].get(config_key, 0)) + 1
                scenario_class = result["metadata"]["scenario_class"]
                state["scenario_counts"][scenario_class] = int(state["scenario_counts"].get(scenario_class, 0)) + 1
                state["valid_samples"] += 1
                accepted_records.append(result)
            else:
                state["rejected_samples"] += 1
                reason = result.get("reason", "unknown")
                state["rejection_reasons"][reason] = int(state["rejection_reasons"].get(reason, 0)) + 1
        if accepted_records:
            next_sample_id = write_records_to_hdf5(h5_path, accepted_records, next_sample_id)
            progress.update(len(accepted_records))
        update_hdf5_report(h5_path, state)
    progress.close()

    if state["valid_samples"] < n_samples:
        raise RuntimeError(f"Only generated {state['valid_samples']} of {n_samples} samples. Rejections: {state['rejection_reasons']}")
    rejection_rate = 100.0 * state["rejected_samples"] / max(state["total_attempted"], 1)
    report = {
        "h5_path": h5_path,
        "total_attempted": int(state["total_attempted"]),
        "valid_samples": int(state["valid_samples"]),
        "rejected_samples": int(state["rejected_samples"]),
        "rejection_rate_percent": float(rejection_rate),
        "powerflow_calls": int(state["powerflow_calls"]),
        "powerflow_calls_per_valid_sample": float(state["powerflow_calls"] / max(state["valid_samples"], 1)),
        "switch_diversity_score": float(len(state["config_counts"]) / max(state["valid_samples"], 1)),
        "scenario_counts": state["scenario_counts"],
        "rejection_reasons": state["rejection_reasons"],
    }
    print(json.dumps(report, indent=2))
    return report


# Run the production default dataset generation.
generation_report = generate_dataset(N_SAMPLES, H5_PATH, BATCH_SIZE, N_JOBS, overwrite=True)


## STEP 12 — Analysis Section (Plotly Only)

The analysis reads scalar labels and metadata only. It does not load node or edge feature tensors for the full dataset into memory.


In [ ]:
def _read_metadata_dataset(dataset):
    raw = dataset[()]
    if isinstance(raw, bytes):
        raw = raw.decode("utf-8")
    return json.loads(raw)


def collect_dataset_analysis(h5_path=H5_PATH):
    rows = []
    load_scales = []
    attrs = {}
    with h5py.File(h5_path, "r") as h5:
        attrs = {key: h5.attrs[key] for key in h5.attrs.keys()}
        sample_names = sorted(name for name in h5.keys() if name.startswith("sample_"))
        for sample_name in tqdm(sample_names, desc="Reading scalar summaries", unit="sample"):
            group = h5[sample_name]
            labels = group["labels"][:]
            global_features = group["global_features"][:]
            metadata = _read_metadata_dataset(group["metadata"])
            load_scales.extend(metadata["load_scaling_vector"])
            rows.append({
                "sample": sample_name,
                "seed": int(metadata["seed"]),
                "scenario_class": metadata["scenario_class"],
                "total_load_mw": float(global_features[0]),
                "total_loss_kw": float(labels[0]),
                "min_voltage_pu": float(labels[1]),
                "max_line_loading_percent": float(labels[2]),
                "voltage_violation_count": int(labels[3]),
                "overload_count": int(labels[4]),
                "constraint_violation_count": int(labels[5]),
                "switch_config_key": metadata["switch_config_key"],
                "topology_id": metadata["topology_id"],
                "risk_score": float(metadata["risk"]["risk_score"]),
                "load_ratio": float(metadata["load_metadata"]["total_load_ratio"]),
            })
    return pd.DataFrame(rows), pd.Series(load_scales, name="load_scaling_factor", dtype=float), attrs


def print_feasibility_report(attrs):
    total = int(attrs.get("total_attempted", 0))
    valid = int(attrs.get("valid_samples", 0))
    rejected = int(attrs.get("rejected_samples", 0))
    powerflow_calls = int(attrs.get("powerflow_calls", 0))
    rejection_rate = 100.0 * rejected / max(total, 1)
    print("Feasibility Report")
    print("-" * 22)
    print(f"Total candidates attempted : {total:,}")
    print(f"Valid samples              : {valid:,}")
    print(f"Rejected candidates        : {rejected:,}")
    print(f"Rejection rate             : {rejection_rate:.2f}%")
    print(f"Power-flow calls           : {powerflow_calls:,}")
    print(f"Power-flow call reduction  : {100.0 * (1.0 - powerflow_calls / max(total, 1)):.2f}%")
    print(f"Rejection reasons          : {json.loads(attrs.get('rejection_reasons', '{}'))}")


def plot_loss_distribution(df):
    fig = px.histogram(df, x="total_loss_kw", color="scenario_class", nbins=60, title="Loss Distribution", labels={"total_loss_kw": "Total loss (kW)"})
    fig.update_layout(bargap=0.02)
    fig.show()


def plot_voltage_distribution(df):
    fig = px.histogram(df, x="min_voltage_pu", color="scenario_class", nbins=60, title="Minimum Voltage Distribution", labels={"min_voltage_pu": "Minimum voltage (pu)"})
    fig.add_vline(x=0.95, line_dash="dash", line_color="orange", annotation_text="0.95 pu")
    fig.show()


def plot_loading_distribution(df):
    fig = px.histogram(df, x="max_line_loading_percent", color="scenario_class", nbins=60, title="Line Loading Distribution", labels={"max_line_loading_percent": "Maximum line loading (%)"})
    fig.add_vline(x=100.0, line_dash="dash", line_color="red", annotation_text="100%")
    fig.show()


def plot_switch_configuration_uniqueness(df):
    counts = df["switch_config_key"].value_counts()
    print(f"Unique switch configurations: {len(counts):,}")
    print(f"Switch diversity score: {len(counts) / max(len(df), 1):.3f}")
    top10 = counts.head(10).reset_index()
    top10.columns = ["switch_config_key", "count"]
    fig = px.bar(top10, x="switch_config_key", y="count", title="Top 10 Switch Configuration Frequencies")
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()


def plot_load_scaling_distribution(load_scale_series):
    fig = px.histogram(load_scale_series.to_frame(), x="load_scaling_factor", nbins=70, title="Load Scaling Distribution", labels={"load_scaling_factor": "Per-bus scaling factor"})
    fig.update_layout(bargap=0.02)
    fig.show()


def plot_correlation_heatmap(df):
    columns = ["total_load_mw", "total_loss_kw", "min_voltage_pu", "max_line_loading_percent", "constraint_violation_count", "risk_score"]
    corr = df[columns].corr()
    fig = go.Figure(data=go.Heatmap(z=corr.values, x=corr.columns, y=corr.index, colorscale="RdBu", zmin=-1, zmax=1, colorbar={"title": "Correlation"}))
    fig.update_layout(title="Correlation Heatmap")
    fig.show()


def plot_constraint_violation_stats(df):
    fig = px.histogram(df, x="constraint_violation_count", color="scenario_class", nbins=30, title="Constraint Violation Statistics", labels={"constraint_violation_count": "Violations per sample"})
    fig.update_layout(bargap=0.02)
    fig.show()


analysis_df, load_scale_series, dataset_attrs = collect_dataset_analysis(H5_PATH)
print_feasibility_report(dataset_attrs)
if len(analysis_df) == 0:
    raise RuntimeError("No valid samples were generated; analysis cannot continue.")
plot_loss_distribution(analysis_df)
plot_voltage_distribution(analysis_df)
plot_loading_distribution(analysis_df)
plot_switch_configuration_uniqueness(analysis_df)
plot_load_scaling_distribution(load_scale_series)
plot_correlation_heatmap(analysis_df)
plot_constraint_violation_stats(analysis_df)
analysis_df.describe(include="all")


## STEP 13 — Final Dataset Summary

This section prints the feature dimensions, label definitions, rejection rate, and flat HDF5 storage schema for downstream Graph Transformer training.


In [ ]:
def print_dataset_summary(h5_path=H5_PATH):
    with h5py.File(h5_path, "r") as h5:
        sample_names = sorted(name for name in h5.keys() if name.startswith("sample_"))
        if len(sample_names) == 0:
            raise RuntimeError("HDF5 file contains no sample groups.")
        first = h5[sample_names[0]]
        total = int(h5.attrs.get("total_attempted", 0))
        valid = int(h5.attrs.get("valid_samples", len(sample_names)))
        rejected = int(h5.attrs.get("rejected_samples", 0))
        rejection_rate = 100.0 * rejected / max(total, 1)
        print("Dataset Ready for Graph Transformer Training")
        print("=" * 52)
        print(f"HDF5 path            : {h5_path}")
        print(f"Total attempts       : {total:,}")
        print(f"Valid samples        : {valid:,}")
        print(f"Rejection rate       : {rejection_rate:.2f}%")
        print(f"Switch diversity     : {float(h5.attrs.get('switch_diversity_score', 0.0)):.3f}")
        print(f"Node feature shape   : {first['node_features'].shape}")
        print(f"Edge feature shape   : {first['edge_features'].shape}")
        print(f"Global feature shape : {first['global_features'].shape}")
        print(f"Node features        : {json.loads(h5.attrs['node_feature_names'])}")
        print(f"Edge features        : {json.loads(h5.attrs['edge_feature_names'])}")
        print(f"Global features      : {json.loads(h5.attrs['global_feature_names'])}")
        print(f"Labels               : {json.loads(h5.attrs['label_names'])}")
        print("")
        print("Label definitions")
        print("- total_loss_kw: total active line losses in kW")
        print("- min_voltage_pu: minimum bus voltage magnitude")
        print("- max_line_loading_percent: maximum line loading percentage")
        print("- voltage_violation_count: buses outside voltage limits")
        print("- overload_count: overloaded lines")
        print("- constraint_violation_count: voltage violations plus overloads")
        print("")
        print("HDF5 storage schema")
        print("/sample_000001/")
        print("    node_features    float32 [num_buses, num_node_features]")
        print("    edge_features    float32 [num_lines, num_edge_features]")
        print("    global_features  float32 [num_global_features]")
        print("    labels           float32 [6]")
        print("    metadata         JSON string")


print_dataset_summary(H5_PATH)
